# Self-play retraining from the existing turn-9 model

Run self-play from the existing `m_9.npz` checkpoint and matching `D_9.npz` dataset. The sequential stage is already complete; this notebook loads those files and does not retrain the turn-9 model. Run the notebook from top to bottom.

The notebook keeps training data and physics caches under `curling_data/`; model checkpoints are saved under `weights/<category>/<run>/checkpoints/`, with records under `stats/`.


In [1]:
import importlib
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while not (repo_root / 'data_generation.py').exists():
    if repo_root.parent == repo_root:
        raise RuntimeError('Could not find repository root')
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import sequential_training as seq
import self_play_training as self_play
import self_play_notebook
from bot import ThrowPerturbations
importlib.reload(seq)
importlib.reload(self_play)
importlib.reload(self_play_notebook)


<module 'self_play_notebook' from '/home/vietafan/curling/self_play_notebook.py'>

In [2]:
# Shared experiment settings
N = 5
seed = 0
grid_size = (10, 10, 4)
robust_top_fraction = 0.05
perturbation_angle_offsets_deg = (0.0, -0.2, 0.2)
perturbation_probabilities = (0.5, 0.25, 0.25)
perturbations = ThrowPerturbations(
    angle_offsets_deg=perturbation_angle_offsets_deg,
    probabilities=perturbation_probabilities,
)

# Source settings for the fixed turn-9 initialization; these do not trigger training
rows_per_model = 20_000
initial_training_steps = 30
random_throw_count = 36 * 4 * 3
policy_comparison_num_sims = 5
state_generator_version = 'turn_based'
data_root = Path('/home/vietafan/curling_data')
sequential_run_name = 'round9_20k'
sequential_run_dir = data_root / 'N5_rows20000_stages2'
legacy_sequential_run_dir = None
sequential_artifact_dir = sequential_run_dir / 'sequential_experiment'
sequential_weights_dir = repo_root / 'weights/sequential_training/round9_20k'
initial_model_path = sequential_weights_dir / 'checkpoints/m_9.npz'
initial_dataset_path = sequential_artifact_dir / 'D_9.npz'
sequential_diagnostic_output_dir = None  # Set a path to save diagnostic checkpoints.

# Self-play settings
num_iterations = 10
num_states_per_iteration = 200
halflife = 5.0
hidden_layer_size = 20
batch_size = 100
training_iterations = 10
learning_rate = 0.5
resume_from = None  # Set to a saved iteration to resume this self-play run.
self_play_run_name = 'round9_2k_v4'  # Fresh outputs after clearing the interrupted attempt.
self_play_artifact_dir = None  # Optional path overrides; None uses the standard layout.
self_play_weights_dir = None

setup = self_play_notebook.make_notebook_setup(locals())
config = setup.config
run_parameters = setup.run_parameters
print(f'self-play retention probability: {config.retention_probability():.4f}')


self-play retention probability: 0.8706


In [3]:
initial_model_path = setup.initial_model_path
initial_dataset_path = setup.initial_dataset_path
assert initial_model_path.is_file(), f'Missing initial model: {initial_model_path}'
assert initial_dataset_path.is_file(), f'Missing initial dataset: {initial_dataset_path}'
print('Using the existing turn-9 initialization:')
print(f'  model: {initial_model_path}')
print(f'  dataset: {initial_dataset_path}')
print(f'  self-play outputs: {setup.self_play_artifact_dir} and {setup.self_play_weights_dir}')


Using the existing turn-9 initialization:
  model: /home/vietafan/curling/weights/sequential_training/round9_20k/checkpoints/m_9.npz
  dataset: /home/vietafan/curling_data/N5_rows20000_stages2/sequential_experiment/D_9.npz
  self-play outputs: /home/vietafan/curling_data/self_play_N5_round9_2k_v4 and /home/vietafan/curling/weights/self_play/round9_2k_v4


In [4]:
results = self_play.run_self_play(
    config=config,
    artifact_dir=setup.self_play_artifact_dir,
    initial_model_path=initial_model_path,
    initial_dataset_path=initial_dataset_path,
    seed=seed,
    resume_from=resume_from,
    weights_dir=setup.self_play_weights_dir,
    notebook_parameters=run_parameters,
)
results


{1: {'new_rows': 600,
  'training_rows': 18075,
  'loss': [0.6543141995886613,
   0.6571586547747956,
   0.6472378526768571,
   0.6407552379996793,
   0.6390212378844534,
   0.6426565184350486,
   0.6310605079846898,
   0.6236882527928052,
   0.6215751231331341,
   0.6207805617956466],
  'r_squared': [0.6667954446061587,
   0.6467101818639651,
   0.6655072519442227,
   0.6732617298940751,
   0.6738762903216207,
   0.6702014959907503,
   0.6785298430839617,
   0.6826918042447496,
   0.6841062386240548,
   0.6847755960403974],
  'mse': [0.35121056957083546,
   0.3723812182113615,
   0.3525683747724504,
   0.3443948532121791,
   0.34374708260310427,
   0.34762045884934173,
   0.3388420568772166,
   0.3344551878319112,
   0.332964318946913,
   0.3322587901789696],
  'epoch_metrics': [{'epoch': 1,
    'cross_entropy': 0.6543141995886613,
    'r_squared': 0.6667954446061587,
    'mse': 0.35121056957083546},
   {'epoch': 2,
    'cross_entropy': 0.6571586547747956,
    'r_squared': 0.646710181

In [5]:
# Each iteration writes D_i.npz and X_i.npz under curling_data, and m_i.npz under weights/<category>/<run>/checkpoints/.
for i, record in results.items():
    print(i, record['new_rows'], record['training_rows'], record['loss'][-1])

1 600 18075 0.6207805617956466
2 600 16343 0.6326604581598354
3 600 14794 0.6337941092438765
4 600 13530 0.6333741934580884
5 600 12348 0.6219122993424239
6 600 11387 0.6101194292363662
7 600 10509 0.6012632964600089
8 600 9763 0.5929070446196906
9 600 9129 0.5812170476724153
10 600 8536 0.5742173945970707
